In [1]:
import pandas as pd
import numpy as np
import json
import re
import gc
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Constants
CURRENT_YEAR = 2026

In [2]:
# Load interactions and ratings
df_watches = pd.read_csv('../data/processed/watches.csv')
df_ratings = pd.read_csv('../data/processed/ratings.csv')

# Load JSON metadata
with open('../metadata/users.json', 'r') as f:
    users_dict = json.load(f)
with open('../metadata/movies.json', 'r') as f:
    movies_dict = json.load(f)

# Convert metadata to DataFrames
df_users = pd.DataFrame.from_dict(users_dict, orient='index')
df_movies = pd.DataFrame.from_dict(movies_dict, orient='index')

print(f"Loaded {len(df_watches)} watches and {len(df_ratings)} ratings.")

Loaded 300482 watches and 210038 ratings.


In [3]:
# Extract year from movie_id using regex: looking for 4 digits at the end
df_watches['release_year'] = df_watches['movie_id'].str.extract(r'(\d{4})$').astype(float)
df_watches['movie_age'] = CURRENT_YEAR - df_watches['release_year']

# Tidy user IDs and gender
df_users['user_id'] = df_users['user_id'].astype(int)
df_users['is_male'] = (df_users['gender'] == 'M').astype(int)

# Align movie IDs for joining
df_movies['movie_id'] = df_movies['id']

In [4]:
# Join watches with ratings and user metadata
df_features = df_watches.merge(df_ratings, on=['user_id', 'movie_id'], how='left')
df_features = df_features.merge(df_users[['user_id', 'age', 'is_male', 'occupation']], on='user_id', how='left')
df_features = df_features.merge(df_movies[['movie_id', 'genres']], on='movie_id', how='left')

# 1. Handle missing ratings (Cold Start strategy)
df_features['rating'] = df_features['rating'].fillna(df_features['rating'].mean())

# 2. User Engagement Score
df_features['user_avg_watch'] = df_features.groupby('user_id')['minutes_watched'].transform('mean')

# 3. Multi-Hot Encode Genres
genre_dummies = df_features['genres'].str.get_dummies(sep=',')
df_features = pd.concat([df_features, genre_dummies], axis=1)

# 4. Memory Management: Downcast to float32
df_features = df_features.select_dtypes(include=['float64']).astype('float32').join(df_features.select_dtypes(exclude=['float64']))

# Clean up memory
del df_watches, df_ratings, genre_dummies
gc.collect()

0

In [5]:
import ast

def clean_genres(genre_str):
    """Parses stringified JSON lists into simple comma-separated strings."""
    try:
        # Convert string representation of list to actual list
        data = ast.literal_eval(genre_str)
        return ",".join([d['name'] for d in data])
    except:
        return "Unknown"

# Apply cleanup before creating dummies
df_features['genres_clean'] = df_features['genres'].apply(clean_genres)
genre_dummies = df_features['genres_clean'].str.get_dummies(sep=',')
df_features = pd.concat([df_features, genre_dummies], axis=1)

In [6]:
# Select numerical features for the model
# Dynamically include all genre columns created in the previous cell
genre_cols = list(df_movies['genres'].str.get_dummies(sep=',').columns)
features = ['age', 'is_male', 'movie_age', 'user_avg_watch', 'rating'] + genre_cols

X = df_features[features].fillna(0)
y = df_features['minutes_watched']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
print("Training model...")
model = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model Training Complete.")

Training model... this may take a minute on a limited VM.
Model Training Complete.


In [8]:
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Model RMSE: {rmse:.2f}")
print(f"Model MAE: {mae:.2f}")

# Feature Importance
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\nTop 5 Features driving recommendations:")
print(importance.head(5))

Model RMSE: 25.87
Model MAE: 17.03

Top 5 Features driving recommendations:
user_avg_watch           0.239411
rating                   0.129160
 'name': 'Comedy'}       0.103992
 'name': 'Animation'}    0.101643
movie_age                0.078399
dtype: float64
